# Education 360° — Business Insights Report
Generate a concise KPI snapshot and export summary tables for the final project report.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_PATH = Path.cwd().parent if Path.cwd().name.lower() == "python" else Path.cwd()
DATA_PATH = PROJECT_PATH / "Dataset"

# Accommodate the source filename spelling found in the project.
FILE_CANDIDATES = {
    "admissions": ["admissions.csv"],
    "branches dim": ["braches dim.csv"],
    "calls": ["calls.csv"],
    "courses dim": ["courses dim.csv"],
    "employees": ["employees.csv"],
    "fees": ["fees.csv"],
    "leads": ["leads.csv"],
    "marketing": ["marketing.csv"],
    "placements": ["placements.csv"],
    "students dim": ["students dim.csv"]
}

def load_table(table_name):
    candidates = FILE_CANDIDATES[table_name]
    for filename in candidates:
        path = DATA_PATH / filename
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(
        f"Could not find {table_name}. Tried: {candidates}. "
        f"Expected dataset folder: {DATA_PATH}"
    )

datasets = {}
for table in FILE_CANDIDATES:
    try:
        datasets[table] = load_table(table)
        print(f"{table:15} {datasets[table].shape}")
    except FileNotFoundError as exc:
        print(exc)

In [ ]:
admissions = datasets["admissions"]
fees = datasets["fees"]
leads = datasets["leads"]
calls = datasets["calls"]
placements = datasets["placements"]
marketing = datasets["marketing"]

def safe_pct(n, d):
    return (n/d*100) if d else np.nan

billed = fees["total fee"].sum()
collected = fees["paid fee"].sum()
remaining = fees["remaining fee"].sum()
converted = leads["lead status"].astype(str).str.strip().str.lower().eq("converted").sum()
effective = calls["Call Category"].astype(str).str.strip().str.lower().eq("effective call").sum()
placed_mask = placements["placement status"].astype(str).str.strip().eq("Placed")
placed_count = placements.loc[placed_mask, "student id"].nunique()

kpis = pd.DataFrame([
    ["Admissions", admissions["admission id"].nunique()],
    ["Total Leads", leads["lead id"].nunique()],
    ["Converted Leads", converted],
    ["Lead Conversion Rate (%)", safe_pct(converted, leads["lead id"].nunique())],
    ["Total Calls", calls["call id"].nunique()],
    ["Effective Calls", effective],
    ["Effective Call Rate (%)", safe_pct(effective, calls["call id"].nunique())],
    ["Placed Students", placed_count],
    ["Placement Rate in Placements Table (%)", safe_pct(placed_count, placements["student id"].nunique())],
    ["Fees Billed", billed],
    ["Fees Collected", collected],
    ["Fees Outstanding", remaining],
    ["Collection Efficiency (%)", safe_pct(collected, billed)]
], columns=["KPI","Value"])
display(kpis)

## Export KPI summary

In [ ]:
OUTPUT_PATH = PROJECT_PATH / "Python" / "outputs"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
kpis.to_csv(OUTPUT_PATH / "education_360_kpi_summary.csv", index=False)
print("Saved:", OUTPUT_PATH / "education_360_kpi_summary.csv")

## Draft insight prompts
Use the outputs from the analysis notebooks to write evidence-based statements:
1. Which lead source has the highest conversion rate?
2. Which branch has the highest admissions and revenue?
3. What proportion of billed fees has been collected?
4. Which placement status group needs the most attention?
5. Which companies and courses account for the largest placement outcomes?

Do not claim causation from descriptive comparisons. Note that 2025 is a partial year if the source date range ends in May 2025.